# Phase 7 (HIGH-SAMPLE variant) — Arrhenius / super-Arrhenius test

**Parallel-safe copy** of `phase7_colab.ipynb` with `--samples-per-prompt 64`
and its OWN output files (`temp_sweep_hi.jsonl`, `arrhenius_report_hi.md`), so
it can run in a second Colab session at the same time as the 24-sample run
without either clobbering the other's Drive file. ~1.5-2 h, a few $ more.

Same OpenRouter account as the other session -> shared rate limits; both may
throttle a little. CPU runtime is enough.

In [ ]:
# 1) Setup
import os
if not os.path.exists('/content/WeirdChat'):
    !git clone --branch claude/repo-published-weights-u71yew https://github.com/Erikiss/WeirdChat /content/WeirdChat
else:
    !git -C /content/WeirdChat pull
%cd /content/WeirdChat/examples/03_deepspec_draft_surprise
%pip install -q -e /content/WeirdChat

from google.colab import drive; drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/weirdspec/data'
os.makedirs(DATA, exist_ok=True); os.environ['DATA'] = DATA
try:
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))
    print('tokens loaded')
except Exception:
    from getpass import getpass
    os.environ.setdefault('OPENROUTER_API_KEY', getpass('OpenRouter key: '))

In [ ]:
# 2a) Clean start — remove any previous HIGH-SAMPLE sweep. Run ONCE.
#     NOTE: touches only temp_sweep_hi.jsonl — it never deletes the other
#     session's temp_sweep.jsonl. Rerun only cell 2 to resume after a drop.
!rm -f $DATA/temp_sweep_hi.jsonl && echo 'previous high-sample sweep cleared'

In [ ]:
# 2) HIGH-SAMPLE temperature sweep (resumable; ~1.5-2 h).
#    64 samples/prompt -> ~3 patterns x 64 = ~192 per behavior per temperature
#    (SE ~2.6% vs ~4% at 24). Writes to its OWN file so it is parallel-safe.
!python phase7_sweep.py --output $DATA/temp_sweep_hi.jsonl \
    --behaviors chemtrails-assertion recommends-drunk-driving \
    --patterns-per-behavior 3 --prompts-per-pattern 6 --samples-per-prompt 64 \
    --temperatures 0.3 0.5 0.7 0.9 1.1 1.3

In [ ]:
# 3) Arrhenius / super-Arrhenius diagnosis (CPU, seconds)
!python phase7_arrhenius.py --sweep $DATA/temp_sweep_hi.jsonl \
    --output $DATA/arrhenius_report_hi.md --key behavior_id

from IPython.display import Markdown, display
display(Markdown(open(os.environ['DATA'] + '/arrhenius_report_hi.md').read()))

In [ ]:
# 4) (Optional) Arrhenius plot: ln(hazard) vs 1/T per behavior
import json, math
import matplotlib.pyplot as plt
from phase7_arrhenius import aggregate, diagnose

rows = [json.loads(l) for l in open(f'{DATA}/temp_sweep_hi.jsonl') if l.strip()]
groups = aggregate(rows, 'behavior_id')
plt.figure(figsize=(6,4))
for name, pts in groups.items():
    r = diagnose(pts, n_boot=200)
    if 'ln_hazard' not in r: continue
    xs = [1.0/t for t in r['temps']]
    plt.plot(xs, r['ln_hazard'], 'o-', label=f"{name} [{r['verdict'].split()[0]}]")
plt.xlabel('1 / T'); plt.ylabel('ln per-token hazard')
plt.title('Arrhenius plot (straight=Arrhenius, concave=super-Arrhenius)')
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## Reading the result

- **SUPER-ARRHENIUS** for language switching (concave plot, `T0>0` inside/near
  the sampled range, VFT preferred by AIC) with **ARRHENIUS** for the fluent
  control would confirm the two-axes picture thermodynamically: surface-form
  tipping has a critical temperature, fluent weirdness does not.
- **INCONCLUSIVE** usually means too few temperatures, or the rate never leaves
  the detection floor/ceiling in the sampled window — widen `--temperatures`
  (add colder points like 0.3, 0.4) and raise `--samples-per-prompt`.
- `fragility` >> 1 is the fragile/super-Arrhenius signature; ~1 is Arrhenius.